In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Library imports and project path setup
# ─────────────────────────────────────────────────────────────────────────────
import sys
import os
import importlib
from datetime import datetime, timedelta

import pandas as pd
import numpy as np


sys.path.append(os.path.abspath('../lib/demand'))  # Environment.py 
sys.path.append(os.path.abspath('../lib/cost'))    # simulationLogic.py, inventoryPolices.py, costTracker.py

import Environment as env_mod
importlib.reload(env_mod)
SimulationConfig = env_mod.SimulationConfig   # dataclass that holds simulation parameters
Simulator        = env_mod.Simulator          # generates the synthetic demand time-series

# StandardInventoryPolicy implements the classical (s, Q) reorder-point / order-quantity
# policy used as the baseline in this study.
from inventoryPolices import StandardInventoryPolicy, InventoryPolicyParams

print(f"Python   : {sys.version.split()[0]}")
import torch
print(f"PyTorch  : {torch.__version__}")
import gymnasium
print(f"Gymnasium: {gymnasium.__version__}")


Python   : 3.11.14
PyTorch  : 2.10.0
Gymnasium: 0.29.1


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# demand data generation
#
# Generates a daily demand time-series for a simulated dealer network and saves
# it to CSV.
# ─────────────────────────────────────────────────────────────────────────────
import os

csv_path = '../data/demand/demand_series.csv'

if os.path.exists(csv_path):
    print(f"Demand CSV found — skipping generation.")
    print(f"  To regenerate: delete '{csv_path}' and re-run this cell.")
else:
    start_time = datetime(2025, 1, 1)
    end_time   = datetime(2031, 1, 1)
    delta_time = 1       # one time step = one calendar day
    seed       = 42      # fixed seed ensures the same data set on every run

    n_dealers     = 1
    n_truck_range = [150, 200]
    n_part_range  = [4, 5]

    cfg    = SimulationConfig(start_time=start_time, end_time=end_time, delta_time=delta_time)
    sim    = Simulator(config=cfg, seed=seed, n_dealers=n_dealers,
                       n_truck_range=n_truck_range, n_part_range=n_part_range)
    events = sim.run()
    print(f"Generation complete: {len(events)} demand events created.")
    print(f"Saved to: {csv_path}")

import pandas as pd
df_check = pd.read_csv(csv_path)
print(f"CSV shape : {df_check.shape}")
print(f"Dealers   : {sorted(df_check['dealer_id'].unique().tolist())}")
print(f"Parts     : {sorted(df_check['part_type'].unique().tolist())}")


Demand CSV found — skipping generation.
  To regenerate: delete '../data/demand/demand_series.csv' and re-run this cell.
CSV shape : (8764, 5)
Dealers   : ['D00']
Parts     : ['type0', 'type1', 'type2', 'type3']


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
#  Gymnasium inventory environment and train/test data split
#
# Defines MultiPartInventoryEnv: a custom Gymnasium environment that simulates
# a spare-parts inventory system for a single dealer with multiple part types.
# All cost parameters are taken directly from lib/cost/costTracker.py so that
# the RL agent and the classical SIP baseline are evaluated under identical
# cost accounting.

import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
import math
import sys
import os
from datetime import timedelta


def load_all_parts_for_dealer(dealer_id: str,
                              csv_path: str = '../data/demand/demand_series.csv'):
    """
    Load the daily demand time-series for every part type of one dealer.

    Returns
    -------
    demand_dict : dict[str, np.ndarray]
        Maps each part-type identifier to a 1-D integer array of daily demands.
    start_date  : datetime
        Calendar date of the first day in the series.
    part_types  : list[str]
        Sorted list of part-type identifiers for this dealer.
    """
    df  = pd.read_csv(csv_path)
    ddf = df[df['dealer_id'] == dealer_id].copy()
    if ddf.empty:
        raise ValueError(f"dealer_id '{dealer_id}' not found in {csv_path}.")

    part_types = sorted(ddf['part_type'].unique().tolist())
    start_date = pd.to_datetime(ddf.sort_values('time')['date'].iloc[0]).to_pydatetime()

    demand_dict = {}
    for pt in part_types:
        arr = (
            ddf[ddf['part_type'] == pt]
            .sort_values('time')['failure']
            .values.astype(int)
        )
        demand_dict[pt] = arr

    print(f"Dealer: {dealer_id} | Start: {start_date.date()} | Parts: {part_types}")
    for pt in part_types:
        arr = demand_dict[pt]
        print(f"  {pt}: {len(arr)} days | total demand={arr.sum()} | avg/day={arr.mean():.4f}")

    return demand_dict, start_date, part_types


class MultiPartInventoryEnv(gym.Env):
    """
    Periodic-review, multi-part inventory environment for spare-parts management.
    The environment closely mirrors the cost accounting in lib/cost/costTracker.py
    and the ordering logic in lib/cost/simulationLogic.py so that the RL agent
    and the classical SIP baseline face an identical cost structure.

    Cost parameters (identical to costTracker.py)
    ─────────────────────────────────────────────
    ORDER_COST     = 100 SEK   fixed cost charged per non-urgent replenishment order
    RUSH_COST      = 165 SEK   fixed cost charged per urgent (emergency) order
    BADWILL_PROXY  =  50 SEK   goodwill/service-level penalty per order line
                                (derived as (1 - 0.95) × 1000 in costTracker.py)
    TRANSPORT_RATE = 0.002     transport cost per unit ordered (2 SEK/kg × 0.001 kg/unit)
    HOLDING_RATE   = 0.15 × 0.13 / 365 ≈ 5.34e-5 SEK/unit/day

    Observation space — 13 features per part (52 features total for 4 parts)
    ─────────────────────────────────────────────────────────────────────────
    Index  Feature                  Description
    ─────  ───────────────────────  ──────────────────────────────────────────────
      0    stock / max_stock        On-hand inventory (normalised to [0, 1])
      1    backorders / max_stock   Accumulated unfulfilled demand (normalised)
      2    inv_pos / max_stock      Inventory position = stock + pipeline − backorders
                                   (can be negative; observation space uses ±inf)
      3    on_order_nu / max_stock  Non-urgent units currently in transit
      4    on_order_u  / max_stock  Urgent units currently in transit
      5    days_to_arrival          Days until next delivery (normalised by lead_time)
      6    demand_rate              This part's mean demand relative to the busiest part
      7    recent_mean_signal       30-day rolling mean demand (normalised)
      8    recent_std_signal        30-day rolling demand std-dev (normalised)
      9    day_of_year              Fractional day-of-year in [0, 1] — seasonality proxy
     10    dyn_rop / max_stock      Dynamic reorder point: μ_30 × L + z × σ_30 × √L
                                   (30-day backward-looking demand history window)
     11    dyn_eoq / max_order      Dynamic economic order quantity (capped at max_order)
                                   (up-to-365-day window for demand rate stability)
     12    sip_signal               Binary: 1 if stock ≤ dyn_rop AND no non-urgent order
                                   is in transit — the SIP would trigger an order now

    Action space
    ────────────
    Continuous Box in [0, max_order] per part, subject to a SIP-triggered hard mask
    applied inside step():
      • When sip_signal = 0 (stock above ROP, or a non-urgent order is already in
        transit), the action is forced to 0 — the agent cannot place an order.
      • When sip_signal = 1, the action is capped at dyn_eoq — the agent decides
        how much to order (up to the EOQ ceiling).
    This reduces the RL problem from "when AND how much" to "how much, given
    the SIP trigger fires", which eliminates the "never-order" local optimum that
    arises when the agent discovers that ordering nothing has lower variance cost
    than occasionally placing orders during exploration.

    Reward
    ──────
    −total_daily_cost  (the negative of all costs incurred that day across all parts).
    Maximising cumulative reward is equivalent to minimising total inventory cost.

    """

    metadata = {'render_modes': ['human']}

    # ── Cost constants — must match costTracker.py exactly ────────────────────
    HOLDING_RATE   = 0.15 * 0.13 / 365.0   # ≈ 5.34e-5 SEK per unit per day
    ORDER_COST     = 100.0                  # fixed charge per non-urgent order
    RUSH_COST      = 165.0                  # fixed charge per urgent order
    BADWILL_PROXY  = 50.0                   # per-order service-level penalty
    TRANSPORT_RATE = 0.002                  # per-unit transport charge

    def __init__(self,
                 demand_dict,
                 start_date,
                 part_types,
                 lead_time=14,
                 urgent_lead=2,
                 initial_stock=120,
                 max_order=5000,
                 use_mask     = True,
                 demand_history_window=30,
):
        """
        Parameters
        ----------
        demand_dict           : dict mapping part_type -> np.ndarray of daily demands
        start_date            : datetime, calendar date of day 0
        part_types            : list of part-type string identifiers
        lead_time             : int, days between placing a non-urgent order and receipt
        urgent_lead           : int, days for an emergency/rush order to arrive
        initial_stock         : float, starting on-hand inventory for every part.
                                Set to 120 so all parts start comfortably above their
                                reorder point (the highest ROP across parts is ≈ 84).
        max_order             : int, upper bound on units per single order
        demand_history_window : int, rolling window size (days) for computing
                                the recent mean and std-dev demand signals, and for
                                the dynamic reorder point.
        """
        super().__init__()
        self.demand_dict = {pt: np.asarray(v, dtype=np.float32) for pt, v in demand_dict.items()}
        self.start_date  = start_date
        self.part_types  = list(part_types)
        self.n_parts     = len(self.part_types)
        self.lead_time   = int(lead_time)
        self.urgent_lead = int(urgent_lead)
        self.initial_stock = float(initial_stock)
        self.max_order   = int(max_order)
        self.use_mask = use_mask
        self.max_stock   = float(max_order * 3)   # warehouse capacity ceiling
        self.T           = len(next(iter(self.demand_dict.values())))
        self.demand_history_window = int(demand_history_window)
        # Pre-computed constants for the dynamic SIP observation features.
        #   ROP = μ × L + z × σ × √L   (standard safety-stock reorder point)
        #   EOQ = √(2 × D × S / H)     (economic order quantity)
        self._HOLDING_COST_ANNUAL = 0.15 * 0.13   # annual holding cost rate × part value
        self._Z95 = 1.6449                         # 95th-percentile z-score (norm.ppf(0.95))

        # Per-part average daily demand and relative demand rate (used in observations)
        self._avg  = {pt: float(self.demand_dict[pt].mean()) for pt in self.part_types}
        max_avg    = max(max(self._avg.values()), 1.0)
        self._rate = {pt: self._avg[pt] / max_avg for pt in self.part_types}

        # 13 observation features per part, concatenated into a flat vector.
        # Inventory position can be negative when backorders exceed available supply,
        # so the observation space bounds are ±inf rather than [0, 1].
        self.obs_per_part = 13
        self.observation_space = spaces.Box(
            low=-np.inf, high=np.inf,
            shape=(self.obs_per_part * self.n_parts,),
            dtype=np.float32,
        )

        # Continuous action: desired order quantity for each part in [0, max_order].
        # The quantity is rounded to an integer inside step().
        self.action_space = spaces.Box(
            low=0.0, high=float(self.max_order),
            shape=(self.n_parts,),
            dtype=np.float32,
        )

    # ─────────────────────────────────────────────────────────────────────────
    def reset(self, seed=None, options=None):
        """Reset the environment to the start of a new episode."""
        super().reset(seed=seed)
        self.day        = 0
        self.stock      = {pt: float(self.initial_stock) for pt in self.part_types}
        self.backorders = {pt: 0.0 for pt in self.part_types}
        self.nu_pipe    = {pt: [] for pt in self.part_types}   # list of (arrival_day, qty)
        self.urg_pipe   = {pt: [] for pt in self.part_types}
        self.demand_hist = {pt: [] for pt in self.part_types}
        return self._obs(), {}

    # ─────────────────────────────────────────────────────────────────────────
    def _deliver_due(self, pipeline, pt, day):
        """Remove and return the total units whose delivery date has been reached."""
        due             = [item for item in pipeline[pt] if item[0] <= day]
        pipeline[pt]    = [item for item in pipeline[pt] if item[0] > day]
        return float(sum(qty for _, qty in due))

    def _forecast_sum(self, pt, start_day, horizon):
        """Sum the historical demand for part pt over a future window [start_day, start_day+horizon)."""
        if horizon <= 0:
            return 0.0
        end_day = min(start_day + horizon, self.T)
        if start_day >= end_day:
            return 0.0
        return float(self.demand_dict[pt][start_day:end_day].sum())

    def _compute_dyn_rop_eoq(self, pt):
        """
        Compute the dynamic ROP and EOQ for a single part from its current
        demand history. Uses the same formulas as _obs() so that the action
        mask applied in step() is always consistent with the sip_signal the
        agent observed at the end of the previous step.

        Returns
        -------
        dyn_rop : float  reorder point in units
        dyn_eoq : float  economic order quantity in units, capped at max_order
        """
        recent  = self.demand_hist[pt][-self.demand_history_window:]
        mu_30   = float(np.mean(recent)) if recent else self._avg[pt]
        std_30  = float(np.std(recent))  if len(recent) > 1 else 0.0
        dyn_rop = mu_30 * self.lead_time + self._Z95 * std_30 * math.sqrt(float(self.lead_time))

        lw_365  = self.demand_hist[pt][-365:]
        mu_365  = float(np.mean(lw_365)) if lw_365 else self._avg[pt]
        D_ann   = mu_365 * 365.0
        dyn_eoq = (math.sqrt(2.0 * D_ann * self.ORDER_COST / self._HOLDING_COST_ANNUAL)
                   if D_ann > 0 else 0.0)
        dyn_eoq = min(dyn_eoq, float(self.max_order))
        return dyn_rop, dyn_eoq

    # ─────────────────────────────────────────────────────────────────────────
    def step(self, action):
        """
        Advance the simulation by one day.

        The daily sequence mirrors simulationLogic.py:
          1. Apply SIP-triggered action mask (hard constraint before processing).
          2. Receive deliveries due today (both non-urgent and urgent pipelines).
          3. Fulfil any outstanding backorders from newly arrived stock.
          4. Satisfy today's demand; any unmet demand becomes new backorders.
          5. If stock ran out, place an automatic urgent replenishment order.
          6. Place the agent's non-urgent order (if quantity > 0 and pipeline is empty).
          7. Charge holding cost on remaining on-hand stock.
        """
        if self.day >= self.T:
            raise RuntimeError('Episode finished. Call reset() before calling step() again.')

        action = np.asarray(action, dtype=np.float32)

        order_qtys = {}
        dyn_eoqs   = {}   # store per-part EOQ for info dict
        for i, pt in enumerate(self.part_types):
            raw_qty = int(np.clip(np.round(float(action[i])), 0, self.max_order))
            dyn_rop, dyn_eoq = self._compute_dyn_rop_eoq(pt)
            dyn_eoqs[pt] = float(dyn_eoq)
            sip_fires = (self.stock[pt] <= dyn_rop and len(self.nu_pipe[pt]) == 0)
            if self.use_mask:
                # Hybrid distributional-interval mask (Gros 2024, Stolz 2024)
                if not sip_fires:
                    order_qtys[pt] = 0                      # mask off: no order
                else:
                    eoq_cap = max(int(dyn_eoq), 1)          # floor at 1 unit
                    order_qtys[pt] = min(raw_qty, eoq_cap)  # cap at dyn_eoq
            else:
                # No mask: agent decides freely (may trigger never-order)
                if len(self.nu_pipe[pt]) == 0:              # only if no order in transit
                    order_qtys[pt] = raw_qty
                else:
                    order_qtys[pt] = 0

        total_cost = 0.0
        info       = {'day': self.day}

        for i, pt in enumerate(self.part_types):
            d = self.day

            # ── Step 2: receive deliveries ─────────────────────────────────
            self.stock[pt] += self._deliver_due(self.nu_pipe, pt, d)
            self.stock[pt] += self._deliver_due(self.urg_pipe, pt, d)

            # ── Step 3: fill outstanding backorders ────────────────────────
            if self.backorders[pt] > 0:
                fulfilled = min(self.stock[pt], self.backorders[pt])
                self.stock[pt]      -= fulfilled
                self.backorders[pt] -= fulfilled

            # ── Step 4: serve today's demand ───────────────────────────────
            demand = float(self.demand_dict[pt][d])


            self.demand_hist[pt].append(demand)

            fulfilled_now = min(self.stock[pt], demand)
            self.stock[pt] -= fulfilled_now
            shortage        = demand - fulfilled_now
            if shortage > 0:
                self.backorders[pt] += shortage

            # ── Step 5: automatic urgent replenishment on stockout ─────────
            # When a stockout occurs an emergency order is placed to cover the
            # shortage plus expected demand until the next scheduled non-urgent
            # delivery arrives (mirrors simulationLogic.py _handle_demand_event).
            rush_cost  = 0.0
            urgent_qty = 0.0
            if shortage > 0:
                next_nonurgent_day = min(
                    [arr_day for arr_day, _ in self.nu_pipe[pt]],
                    default=d + self.lead_time,
                )
                days_until_arrival   = max(0, next_nonurgent_day - d)
                expected_until_arr   = self._forecast_sum(pt, d + 1, days_until_arrival)
                urgent_qty           = shortage + expected_until_arr
                if urgent_qty > 0:
                    self.urg_pipe[pt].append((d + self.urgent_lead, float(urgent_qty)))
                    rush_cost = self.RUSH_COST + self.BADWILL_PROXY + self.TRANSPORT_RATE * urgent_qty

            # ── Step 6: place agent's non-urgent order ─────────────────────
            order_cost   = 0.0
            order_placed = False
            order_qty    = 0.0
            qty          = order_qtys[pt]   # already masked and capped by Step 1

            # Guard: do not stack a second non-urgent order if one is already in transit.
            # (The SIP mask in Step 1 already enforces this, but the check is kept
            # here as a safety net in case of edge cases during the delivery phase.)
            has_nonurgent_in_transit = len(self.nu_pipe[pt]) > 0

            if qty > 0 and not has_nonurgent_in_transit:
                # Inventory-position cap: never order beyond the warehouse capacity.
                on_order_all = (sum(q for _, q in self.nu_pipe[pt])
                                + sum(q for _, q in self.urg_pipe[pt]))
                inv_pos_now  = self.stock[pt] + on_order_all - self.backorders[pt]
                qty          = min(qty, max(0, int(self.max_stock - inv_pos_now)))

            if qty > 0 and not has_nonurgent_in_transit:
                self.nu_pipe[pt].append((d + self.lead_time, float(qty)))
                order_placed = True
                order_qty    = float(qty)
                order_cost   = self.ORDER_COST + self.BADWILL_PROXY + self.TRANSPORT_RATE * order_qty

            # ── Step 7: holding cost on end-of-day stock ───────────────────
            holding  = self.HOLDING_RATE * self.stock[pt]
            day_cost = rush_cost + order_cost + holding
            total_cost += day_cost

            on_order_nu = float(sum(q for _, q in self.nu_pipe[pt]))
            on_order_u  = float(sum(q for _, q in self.urg_pipe[pt]))

            info[pt] = {
                'stock':              float(self.stock[pt]),
                'backorders':         float(self.backorders[pt]),
                'on_order':           float(on_order_nu + on_order_u),
                'on_order_nonurgent': float(on_order_nu),
                'on_order_urgent':    float(on_order_u),
                'demand':             float(demand),
                'shortage':           float(shortage),
                'urgent_qty':         float(urgent_qty),
                'order_placed':       bool(order_placed),
                'order_qty':          float(order_qty),
                'day_cost':           float(day_cost),
                'ordering':           float(order_cost),
                'rush':               float(rush_cost),
                'holding':            float(holding),
                'dyn_eoq':            dyn_eoqs[pt],
            }

        self.day += 1
        terminated         = self.day >= self.T
        info['total_day_cost'] = float(total_cost)
        # Reward is the negative of cost: the agent maximises reward by minimising cost.
        return self._obs(), -float(total_cost), terminated, False, info

    # ─────────────────────────────────────────────────────────────────────────
    def _obs(self):
        """
        Construct the 13 × n_parts observation vector for the current state.

        Dynamic SIP features (indices 10–12) are recomputed every step using
        backward-looking rolling demand windows:
          - ROP uses a 30-day window (demand_history_window)
          - EOQ uses an up-to-365-day window for a more stable annual estimate
        """
        blocks = []
        for pt in self.part_types:
            stock       = float(self.stock[pt])
            backorders  = float(self.backorders[pt])
            on_order_nu = float(sum(q for _, q in self.nu_pipe[pt]))
            on_order_u  = float(sum(q for _, q in self.urg_pipe[pt]))

            # Inventory position can be negative when backorders exceed available supply.
            inv_pos = stock + on_order_nu + on_order_u - backorders

            # Days until the next scheduled delivery (from either pipeline).
            arrivals        = [day for day, _ in self.nu_pipe[pt] + self.urg_pipe[pt]]
            days_to_arrival = float(max(0, min(arrivals) - self.day)) if arrivals else 0.0

            # 30-day rolling demand statistics for the mean/std signals and for ROP.
            recent      = self.demand_hist[pt][-self.demand_history_window:]
            recent_mean = float(np.mean(recent)) if recent else 0.0
            recent_std  = float(np.std(recent))  if len(recent) > 1 else 0.0

            # Fractional day-of-year in [0, 1] — captures seasonal demand patterns.
            from datetime import timedelta as _td
            cur_date = self.start_date + _td(days=self.day)
            doy      = cur_date.timetuple().tm_yday / 366.0

            # ── Dynamic ROP and EOQ — shared helper to stay in sync with step() ──
            dyn_rop, dyn_eoq = self._compute_dyn_rop_eoq(pt)

            # sip_signal = 1 when the classical SIP policy would trigger an order.
            dyn_sip = 1.0 if (stock <= dyn_rop and on_order_nu < 1.0) else 0.0

            blocks.append(np.array([
                stock        / self.max_stock,                          # 0
                backorders   / self.max_stock,                          # 1
                inv_pos      / self.max_stock,                          # 2  
                on_order_nu  / self.max_stock,                          # 3
                on_order_u   / self.max_stock,                          # 4
                days_to_arrival / float(max(self.lead_time, 1)),        # 5
                self._rate[pt],                                         # 6
                recent_mean / max(self._avg[pt] * 2.0, 1.0),           # 7
                recent_std  / max(self._avg[pt] * 2.0, 1.0),           # 8
                doy,                                                    # 9
                dyn_rop / self.max_stock,                               # 10
                dyn_eoq / float(self.max_order),                        # 11
                dyn_sip,                                                # 12
            ], dtype=np.float32))

        return np.concatenate(blocks).astype(np.float32)

    # ─────────────────────────────────────────────────────────────────────────
    def render(self):
        """Print a one-line stock/pipeline summary for each part (human-readable)."""
        print(f'Day {self.day}')
        for pt in self.part_types:
            print(
                f"  {pt}: stock={self.stock[pt]:.1f}  backorders={self.backorders[pt]:.1f}"
                f"  nu_pipe={self.nu_pipe[pt]}  urg_pipe={self.urg_pipe[pt]}"
            )


# ─────────────────────────────────────────────────────────────────────────────
# 80 / 20  train / test split
# ─────────────────────────────────────────────────────────────────────────────
demand_dict, start_date, part_types = load_all_parts_for_dealer('D00')

T_total = len(demand_dict[part_types[0]])
SPLIT   = int(T_total * 0.80)

train_demand_dict = {pt: arr[:SPLIT] for pt, arr in demand_dict.items()}
test_demand_dict  = {pt: arr[SPLIT:] for pt, arr in demand_dict.items()}
train_start_date  = start_date
test_start_date   = start_date + timedelta(days=SPLIT)
T_train           = SPLIT
T_test            = T_total - SPLIT

print(f'\nTrain / test split:')
print(f'  Train : {T_train} days  ({train_start_date.date()} → {(train_start_date + timedelta(days=T_train)).date()})')
print(f'  Test  : {T_test}  days  ({test_start_date.date()} → {(test_start_date + timedelta(days=T_test)).date()})')
print(f'\nObservation size  : {13 * len(part_types)}  ({13} features × {len(part_types)} parts)')
print(f'Action space      : SIP-masked continuous order quantity in [0, EOQ] per part')
print(f'Cost constants    : ORDER={MultiPartInventoryEnv.ORDER_COST:.0f} SEK  '
      f'RUSH={MultiPartInventoryEnv.RUSH_COST:.0f} SEK  '
      f'BADWILL={MultiPartInventoryEnv.BADWILL_PROXY:.0f} SEK  '
      f'HOLD={MultiPartInventoryEnv.HOLDING_RATE:.2e} SEK/unit/day  '
      f'TRANSPORT={MultiPartInventoryEnv.TRANSPORT_RATE} SEK/unit')

# ── Compute static SIP parameters from the training set ──────────────────────
# These are computed once and used for display/verification only.
# The environment's dynamic ROP/EOQ features are re-computed every step
# from rolling demand windows (see _obs()), so these static values are
# only used in the summary printout below and nowhere else in training/eval.
try:
    _pol = StandardInventoryPolicy(
        InventoryPolicyParams(lead_time=14, service_level=0.95, review_period=1)
    )
    _rop = {pt: float(_pol.calculate_reorder_point(train_demand_dict[pt])) for pt in part_types}
    _eoq = {pt: min(float(_pol.calculate_order_quantity(train_demand_dict[pt])), 200.0)
            for pt in part_types}
except Exception:
    # Fallback in case inventoryPolices.py is unavailable
    _z   = 1.6449
    _rop, _eoq = {}, {}
    for pt in part_types:
        arr      = train_demand_dict[pt]
        mu, std  = arr.mean(), arr.std()
        _rop[pt] = float(round(mu * 14 + _z * std * math.sqrt(14)))
        _eoq[pt] = 5000.0

print('\nStatic SIP parameters (fitted on training data, for reference):')
for pt in part_types:
    print(f'  {pt}: ROP = {_rop[pt]:.1f}   EOQ = {_eoq[pt]:.0f}   (initial_stock = 120 > max ROP = 83)')


Dealer: D00 | Start: 2025-01-02 | Parts: ['type0', 'type1', 'type2', 'type3']
  type0: 2191 days | total demand=2747 | avg/day=1.2538
  type1: 2191 days | total demand=5293 | avg/day=2.4158
  type2: 2191 days | total demand=10494 | avg/day=4.7896
  type3: 2191 days | total demand=4071 | avg/day=1.8581

Train / test split:
  Train : 1752 days  (2025-01-02 → 2029-10-20)
  Test  : 439  days  (2029-10-20 → 2031-01-02)

Observation size  : 52  (13 features × 4 parts)
Action space      : SIP-masked continuous order quantity in [0, EOQ] per part
Cost constants    : ORDER=100 SEK  RUSH=165 SEK  BADWILL=50 SEK  HOLD=5.34e-05 SEK/unit/day  TRANSPORT=0.002 SEK/unit

Static SIP parameters (fitted on training data, for reference):
  type0: ROP = 23.0   EOQ = 200   (initial_stock = 120 > max ROP = 83)
  type1: ROP = 43.0   EOQ = 200   (initial_stock = 120 > max ROP = 83)
  type2: ROP = 83.0   EOQ = 200   (initial_stock = 120 > max ROP = 83)
  type3: ROP = 35.0   EOQ = 200   (initial_stock = 120 > ma

In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4 — Shared hyperparameters and Proximal Policy Optimisation (PPO)
#
# All shared constants are defined once here and referenced by every training
# cell (4–7).  Changing a value in one place updates all four algorithms,
# guaranteeing that the performance comparison is controlled.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='stable_baselines3')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3'])

import numpy as np
from stable_baselines3 import PPO
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.callbacks import BaseCallback


# ═════════════════════════════════════════════════════════════════════════════
# SHARED HYPERPARAMETERS — identical across PPO, SAC, A2C, and TD3
# ═════════════════════════════════════════════════════════════════════════════
LR         = 3e-4        # Adam learning rate
GAMMA      = 0.995       # discount factor — high value preserves signals across the 14-day lead time
BATCH_SIZE = 256         # minibatch size for gradient updates
NET_ARCH   = [256, 256]  # MLP: two hidden layers of 256 units with ReLU activations
DEVICE     = 'cpu'       # all models trained on CPU (single-environment setting)

# On-policy shared: PPO and A2C
N_STEPS    = 30     # rollout length in days; exceeds the 14-day lead time so cost
                    # consequences of an order appear within the same rollout buffer
GAE_LAMBDA = 0.97   # GAE trade-off between bias (λ→0) and variance (λ→1)
ENT_COEF   = 0.05  # small entropy bonus; the SIP mask removes the need for entropy-
                    # driven exploration to escape the never-order local optimum
MAX_GRAD   = 0.5    # gradient clipping threshold

# Off-policy shared: SAC and TD3
BUFFER     = 500_000  # replay buffer capacity (~285 full training episodes)
TAU        = 0.005    # Polyak averaging coefficient for target network updates
TRAIN_FREQ = T_train  # collect one full episode before each gradient phase
GRAD_STEPS = 128      # gradient steps performed per training phase
LRN_START  = T_train  # fill the buffer with one episode before training begins
# ═════════════════════════════════════════════════════════════════════════════


# ─────────────────────────────────────────────────────────────────────────────
# EpisodeProgressCallback
# ─────────────────────────────────────────────────────────────────────────────
class EpisodeProgressCallback(BaseCallback):
    """
    Logs mean episode reward at regular intervals during training.

    Parameters
    ----------
    total_episodes : int   total number of training episodes
    algo_name      : str   algorithm label printed in the progress line
    print_every    : int   print after every this many completed episodes
    """
    def __init__(self, total_episodes, algo_name='ALGO', print_every=200):
        super().__init__()
        self.total_episodes  = total_episodes
        self.algo_name       = algo_name
        self.print_every     = print_every
        self.ep_count        = 0
        self._recent_rewards = []
        self._t0             = None

    def _on_training_start(self):
        self._t0 = time.time()

    def _on_step(self) -> bool:
        for info in self.locals.get('infos', []):
            if 'episode' in info:
                self.ep_count += 1
                self._recent_rewards.append(info['episode']['r'])
                if self.ep_count % self.print_every == 0:
                    elapsed = time.time() - self._t0
                    eta     = elapsed / self.ep_count * (self.total_episodes - self.ep_count)
                    window  = self._recent_rewards[-self.print_every:]
                    mean_r  = sum(window) / len(window)
                    print(f'  {self.algo_name}  ep {self.ep_count:>5}/{self.total_episodes}'
                          f'  mean_reward {mean_r:>12,.0f}'
                          f'  elapsed {elapsed/60:.1f}m  ETA {eta/60:.1f}m')
        return True


def make_train_env_ppo():
    """Create a fresh training environment wrapped with Monitor for episode tracking."""
    return Monitor(MultiPartInventoryEnv(
        demand_dict  = train_demand_dict,
        start_date   = train_start_date,
        part_types   = part_types,
        lead_time    = 14,
        urgent_lead  = 2,
        initial_stock= 120,
        max_order    = 5000,
        use_mask     = True,
    ))


train_env_ppo   = make_train_env_ppo()
PPO_EPISODES    = 1500
TRAIN_STEPS_PPO = T_train * PPO_EPISODES

print(f'PPO training: {PPO_EPISODES} episodes × {T_train} days = {TRAIN_STEPS_PPO:,} steps')

ppo_model = PPO(
    'MlpPolicy',
    train_env_ppo,
    verbose       = 0,
    # ── shared ──────────────────────────────
    learning_rate = LR,
    gamma         = GAMMA,
    batch_size    = BATCH_SIZE,
    policy_kwargs = dict(net_arch=NET_ARCH),
    device        = DEVICE,
    # ── on-policy shared (PPO + A2C) ────────
    n_steps       = N_STEPS,
    gae_lambda    = GAE_LAMBDA,
    ent_coef      = ENT_COEF,
    max_grad_norm = MAX_GRAD,
    # ── PPO-specific ────────────────────────
    n_epochs      = 4,      # gradient passes over each rollout buffer before discarding
    seed          = 42,     # fixed seed for reproducible weight init and training
    clip_range    = 0.2,    # PPO trust-region clip on the probability ratio r_t(φ)
)

t_start = time.time()
ppo_model.learn(
    total_timesteps = TRAIN_STEPS_PPO,
    callback        = EpisodeProgressCallback(PPO_EPISODES, algo_name='PPO'),
)
t_ppo = time.time() - t_start

ep_rewards_ppo = train_env_ppo.get_episode_rewards()
print(f'\nPPO training complete in {t_ppo/60:.1f} min')
print(f'Last-20-episode mean cost: {-np.mean(ep_rewards_ppo[-20:]):,.0f} SEK/episode')

Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
Users of this version of Gym should be able to simply replace 'import gym' with 'import gymnasium as gym' in the vast majority of cases.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.


PPO training: 2000 episodes × 1752 days = 3,504,000 steps
  PPO  ep   200/2000  mean_reward     -186,786  elapsed 2.8m  ETA 25.3m
  PPO  ep   400/2000  mean_reward     -170,517  elapsed 5.6m  ETA 22.4m
  PPO  ep   600/2000  mean_reward     -149,260  elapsed 8.4m  ETA 19.6m
  PPO  ep   800/2000  mean_reward     -124,057  elapsed 11.2m  ETA 16.7m
  PPO  ep  1000/2000  mean_reward     -114,249  elapsed 13.9m  ETA 13.9m
  PPO  ep  1200/2000  mean_reward     -105,223  elapsed 16.7m  ETA 11.2m
  PPO  ep  1400/2000  mean_reward      -94,743  elapsed 19.9m  ETA 8.5m
  PPO  ep  1600/2000  mean_reward      -80,802  elapsed 24.4m  ETA 6.1m
  PPO  ep  1800/2000  mean_reward      -66,181  elapsed 28.4m  ETA 3.2m
  PPO  ep  2000/2000  mean_reward      -54,007  elapsed 32.0m  ETA 0.0m

PPO training complete in 32.0 min
Last-20-episode mean cost: 49,311 SEK/episode


In [5]:
# ───'──────────────────────────────────────────────────────────────────────────
# Cell 5 — Soft Actor-Critic (SAC)
#
# Shared constants (LR, GAMMA, BATCH_SIZE, NET_ARCH, DEVICE, BUFFER, TAU,
# TRAIN_FREQ, GRAD_STEPS, LRN_START) are defined in Cell 4.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='stable_baselines3')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3'])

import numpy as np
from stable_baselines3 import SAC
from stable_baselines3.common.monitor import Monitor


def make_train_env_sac():
    """Create a fresh training environment wrapped with Monitor for episode tracking."""
    return Monitor(MultiPartInventoryEnv(
        demand_dict  = train_demand_dict,
        start_date   = train_start_date,
        part_types   = part_types,
        lead_time    = 14,
        urgent_lead  = 2,
        initial_stock= 120,
        max_order    = 5000,
        use_mask     = True,
    ))


train_env_sac   = make_train_env_sac()
SAC_EPISODES    = 1500
TRAIN_STEPS_SAC = T_train * SAC_EPISODES

print(f'SAC training: {SAC_EPISODES} episodes × {T_train} days = {TRAIN_STEPS_SAC:,} steps')

sac_model = SAC(
    'MlpPolicy',
    train_env_sac,
    verbose          = 0,
    # ── shared ──────────────────────────────
    learning_rate    = LR,
    gamma            = GAMMA,
    batch_size       = BATCH_SIZE,
    policy_kwargs    = dict(net_arch=NET_ARCH),
    device           = DEVICE,
    # ── off-policy shared (SAC + TD3) ───────
    buffer_size      = BUFFER,
    tau              = TAU,
    train_freq       = TRAIN_FREQ,
    gradient_steps   = GRAD_STEPS,
    learning_starts  = LRN_START,
    # ── SAC-specific ────────────────────────
    seed             = 42,      # fixed seed for reproducible weight init and training
    ent_coef         = 'auto',  # temperature α tuned to target entropy −|A| = −4 nats
)

t_start = time.time()
sac_model.learn(
    total_timesteps = TRAIN_STEPS_SAC,
    callback        = EpisodeProgressCallback(SAC_EPISODES, algo_name='SAC'),
)
t_sac = time.time() - t_start

ep_rewards_sac = train_env_sac.get_episode_rewards()
print(f'\nSAC training complete in {t_sac/60:.1f} min')
print(f'Last-20-episode mean cost: {-np.mean(ep_rewards_sac[-20:]):,.0f} SEK/episode')

SAC training: 2000 episodes × 1752 days = 3,504,000 steps
  SAC  ep   200/2000  mean_reward      -11,828  elapsed 4.8m  ETA 43.0m
  SAC  ep   400/2000  mean_reward       -2,843  elapsed 9.0m  ETA 35.9m
  SAC  ep   600/2000  mean_reward       -2,843  elapsed 12.9m  ETA 30.2m
  SAC  ep   800/2000  mean_reward       -2,843  elapsed 16.9m  ETA 25.3m
  SAC  ep  1000/2000  mean_reward       -2,843  elapsed 20.8m  ETA 20.8m
  SAC  ep  1200/2000  mean_reward       -2,843  elapsed 24.8m  ETA 16.5m
  SAC  ep  1400/2000  mean_reward       -2,843  elapsed 28.8m  ETA 12.3m
  SAC  ep  1600/2000  mean_reward       -2,843  elapsed 32.8m  ETA 8.2m
  SAC  ep  1800/2000  mean_reward       -2,843  elapsed 37.2m  ETA 4.1m
  SAC  ep  2000/2000  mean_reward       -2,843  elapsed 42.1m  ETA 0.0m

SAC training complete in 42.1 min
Last-20-episode mean cost: 2,843 SEK/episode


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6 — Advantage Actor-Critic (A2C)
#
# Shared constants (LR, GAMMA, NET_ARCH, DEVICE, N_STEPS, GAE_LAMBDA,
# ENT_COEF, MAX_GRAD) are defined in Cell 4.
# A2C does not accept a batch_size parameter: its effective minibatch is
# determined by n_steps (rollout length), matching the N_STEPS constant.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='stable_baselines3')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3'])

import numpy as np
from stable_baselines3 import A2C
from stable_baselines3.common.monitor import Monitor


def make_train_env_a2c():
    """Create a fresh training environment wrapped with Monitor for episode tracking."""
    return Monitor(MultiPartInventoryEnv(
        demand_dict  = train_demand_dict,
        start_date   = train_start_date,
        part_types   = part_types,
        lead_time    = 14,
        urgent_lead  = 2,
        initial_stock= 120,
        max_order    = 5000,
        use_mask     = True,
    ))


train_env_a2c   = make_train_env_a2c()
A2C_EPISODES    = 1500
TRAIN_STEPS_A2C = T_train * A2C_EPISODES

print(f'A2C training: {A2C_EPISODES} episodes × {T_train} days = {TRAIN_STEPS_A2C:,} steps')

a2c_model = A2C(
    'MlpPolicy',
    train_env_a2c,
    verbose       = 0,
    # ── shared ──────────────────────────────
    learning_rate = LR,
    gamma         = GAMMA,
    policy_kwargs = dict(net_arch=NET_ARCH),
    device        = DEVICE,
    # ── on-policy shared (PPO + A2C) ────────
    n_steps       = N_STEPS,
    gae_lambda    = GAE_LAMBDA,
    ent_coef      = ENT_COEF,
    max_grad_norm = MAX_GRAD,
    # ── A2C-specific ────────────────────────
    seed          = 42,   # fixed seed for reproducible weight init and training
    vf_coef       = 0.5,  # weight of the value-function loss relative to the policy loss
)

t_start = time.time()
a2c_model.learn(
    total_timesteps = TRAIN_STEPS_A2C,
    callback        = EpisodeProgressCallback(A2C_EPISODES, algo_name='A2C'),
)
t_a2c = time.time() - t_start

ep_rewards_a2c = train_env_a2c.get_episode_rewards()
print(f'\nA2C training complete in {t_a2c/60:.1f} min')
print(f'Last-20-episode mean cost: {-np.mean(ep_rewards_a2c[-20:]):,.0f} SEK/episode')

A2C training: 2000 episodes × 1752 days = 3,504,000 steps
  A2C  ep   200/2000  mean_reward     -119,158  elapsed 2.2m  ETA 19.8m
  A2C  ep   400/2000  mean_reward     -117,509  elapsed 4.4m  ETA 17.6m
  A2C  ep   600/2000  mean_reward     -129,260  elapsed 6.8m  ETA 15.8m
  A2C  ep   800/2000  mean_reward     -129,908  elapsed 9.1m  ETA 13.7m
  A2C  ep  1000/2000  mean_reward     -134,447  elapsed 11.4m  ETA 11.4m
  A2C  ep  1200/2000  mean_reward     -132,180  elapsed 13.8m  ETA 9.2m
  A2C  ep  1400/2000  mean_reward     -131,797  elapsed 16.1m  ETA 6.9m
  A2C  ep  1600/2000  mean_reward     -122,455  elapsed 18.5m  ETA 4.6m
  A2C  ep  1800/2000  mean_reward     -122,577  elapsed 20.8m  ETA 2.3m
  A2C  ep  2000/2000  mean_reward     -124,685  elapsed 23.1m  ETA 0.0m

A2C training complete in 23.1 min
Last-20-episode mean cost: 144,601 SEK/episode


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7 — Twin Delayed DDPG (TD3)
#
# Shared constants (LR, GAMMA, BATCH_SIZE, NET_ARCH, DEVICE, BUFFER, TAU,
# TRAIN_FREQ, GRAD_STEPS, LRN_START) are defined in Cell 4.
# ─────────────────────────────────────────────────────────────────────────────
import subprocess, sys, os, time, warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='stable_baselines3')
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'stable-baselines3'])

import numpy as np
from stable_baselines3 import TD3
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.noise import NormalActionNoise


def make_train_env_td3():
    """Create a fresh training environment wrapped with Monitor for episode tracking."""
    return Monitor(MultiPartInventoryEnv(
        demand_dict  = train_demand_dict,
        start_date   = train_start_date,
        part_types   = part_types,
        lead_time    = 14,
        urgent_lead  = 2,
        initial_stock= 120,
        max_order    = 5000,
        use_mask     = True,
    ))


train_env_td3   = make_train_env_td3()
TD3_EPISODES    = 1500
TRAIN_STEPS_TD3 = T_train * TD3_EPISODES

# TD3 uses a deterministic policy; exploration requires additive Gaussian noise.
# σ = 20 ≈ 10 % of Q_max = 200, a standard choice for continuous control tasks.
n_actions    = train_env_td3.action_space.shape[0]
action_noise = NormalActionNoise(
    mean  = np.zeros(n_actions),
    sigma = 20.0 * np.ones(n_actions),
)

print(f'TD3 training: {TD3_EPISODES} episodes × {T_train} days = {TRAIN_STEPS_TD3:,} steps')

td3_model = TD3(
    'MlpPolicy',
    train_env_td3,
    verbose          = 0,
    # ── shared ──────────────────────────────
    learning_rate    = LR,
    gamma            = GAMMA,
    batch_size       = BATCH_SIZE,
    policy_kwargs    = dict(net_arch=NET_ARCH),
    device           = DEVICE,
    # ── off-policy shared (SAC + TD3) ───────
    buffer_size      = BUFFER,
    tau              = TAU,
    train_freq       = TRAIN_FREQ,
    gradient_steps   = GRAD_STEPS,
    learning_starts  = LRN_START,
    # ── TD3-specific ────────────────────────
    action_noise     = action_noise,  # Gaussian exploration noise N(0, 20²)
    seed             = 42,            # fixed seed for reproducible weight init and training
    policy_delay     = 2,             # actor updated every 2 critic steps
)

t_start = time.time()
td3_model.learn(
    total_timesteps = TRAIN_STEPS_TD3,
    callback        = EpisodeProgressCallback(TD3_EPISODES, algo_name='TD3'),
)
t_td3 = time.time() - t_start

ep_rewards_td3 = train_env_td3.get_episode_rewards()
print(f'\nTD3 training complete in {t_td3/60:.1f} min')
print(f'Last-20-episode mean cost: {-np.mean(ep_rewards_td3[-20:]):,.0f} SEK/episode')

TD3 training: 2000 episodes × 1752 days = 3,504,000 steps
  TD3  ep   200/2000  mean_reward       -3,616  elapsed 3.5m  ETA 31.3m
  TD3  ep   400/2000  mean_reward       -3,493  elapsed 7.3m  ETA 29.2m
  TD3  ep   600/2000  mean_reward       -3,557  elapsed 11.4m  ETA 26.7m
  TD3  ep   800/2000  mean_reward       -3,527  elapsed 15.4m  ETA 23.2m
  TD3  ep  1000/2000  mean_reward       -3,498  elapsed 19.3m  ETA 19.3m
  TD3  ep  1200/2000  mean_reward       -3,448  elapsed 23.1m  ETA 15.4m
  TD3  ep  1400/2000  mean_reward       -3,511  elapsed 26.9m  ETA 11.5m
  TD3  ep  1600/2000  mean_reward       -3,487  elapsed 30.9m  ETA 7.7m
  TD3  ep  1800/2000  mean_reward       -3,477  elapsed 34.7m  ETA 3.9m
  TD3  ep  2000/2000  mean_reward       -3,419  elapsed 38.4m  ETA 0.0m

TD3 training complete in 38.4 min
Last-20-episode mean cost: 3,366 SEK/episode


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8 — Policy evaluation on the held-out test set
#
# Evaluates every model trained in the current kernel session against the
# classical SIP (s, Q) baseline on the same held-out test environment.
# Each plot is saved both as a high-resolution PNG (overview) and as an
# individual PDF (LaTeX-ready, one file per figure).
# ─────────────────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore', category=DeprecationWarning)
warnings.filterwarnings('ignore', category=UserWarning, module='stable_baselines3')

from stable_baselines3 import PPO, SAC, A2C, TD3



# ── Identify models trained in this session ───────────────────────────────────
print('Models available for evaluation:')
_available = {}
for _name, _var in [('PPO', 'ppo_model'),
                    ('SAC', 'sac_model'),
                    ('A2C', 'a2c_model'),
                    ('TD3', 'td3_model')]:
    _m = globals().get(_var)
    if _m is not None:
        print(f'  {_name}: trained this session ✓')
        _available[_name] = _m
    else:
        print(f'  {_name}: not trained this session — skipped')


# ── Test environment factory ──────────────────────────────────────────────────
def make_test_env():
    """Return a fresh deterministic test environment (test_demand_dict, test_start_date)."""
    return MultiPartInventoryEnv(
        demand_dict  = test_demand_dict,
        start_date   = test_start_date,
        part_types   = part_types,
        lead_time    = 14,
        urgent_lead  = 2,
        initial_stock= 120,
        max_order    = 5000,
        use_mask     = True,
    )


# ── Episode runner ────────────────────────────────────────────────────────────
def run_episode(env, action_fn):
    """
    Run one complete episode and collect per-step info dicts.

    Parameters
    ----------
    env       : MultiPartInventoryEnv  fresh (not yet reset) test environment
    action_fn : callable(obs, env) -> np.ndarray  policy to evaluate

    Returns
    -------
    list[dict]  one info dict per day in the episode
    """
    obs, _  = env.reset()
    rows, done = [], False
    while not done:
        action = np.asarray(action_fn(obs, env), dtype=np.float32)
        obs, reward, term, trunc, info = env.step(action)
        info['action'] = action.tolist()
        info['reward'] = float(reward)
        rows.append(info)
        done = term or trunc
    return rows


# ── Observation-space compatibility check ────────────────────────────────────
def check_model(model, expected_obs_size, name):
    """Warn and skip if a model was trained on a different observation layout."""
    model_obs = model.observation_space.shape[0]
    if model_obs != expected_obs_size:
        print(f'  WARNING — {name} obs size mismatch: '              f'model={model_obs}, env={expected_obs_size}. Re-run training cell.')
        return False
    return True


# ── SIP (s, Q) baseline policy ────────────────────────────────────────────────
def baseline_action(obs, env):
    """
    Classical dynamic reorder-point / EOQ policy.

    Reads dyn_rop (obs index 10) and dyn_eoq (obs index 11) from the observation
    vector, which are recomputed each step from a 30-day rolling demand window.
    Places an order of size dyn_eoq whenever on-hand stock ≤ dyn_rop and no
    non-urgent order is in transit. The SIP action mask in step() does not alter
    this baseline because the baseline already orders at exactly the same
    trigger condition as the mask.
    """
    action    = np.zeros(env.n_parts, dtype=np.float32)
    max_stock = float(env.max_stock)
    max_order = float(env.max_order)
    for i in range(env.n_parts):
        blk  = obs[i * env.obs_per_part : (i + 1) * env.obs_per_part]
        stock              = float(blk[0])  * max_stock
        on_order_nonurgent = float(blk[3])  * max_stock
        dyn_rop            = float(blk[10]) * max_stock
        dyn_eoq            = float(blk[11]) * max_order
        if stock <= dyn_rop and on_order_nonurgent < 1.0:
            action[i] = min(dyn_eoq, max_order)
    return action


# ── RL policy wrappers ────────────────────────────────────────────────────────
# deterministic=True: use the policy mean — no exploration noise at test time.
def ppo_action(obs, env):
    a, _ = ppo_model.predict(obs, deterministic=True); return a

def sac_action(obs, env):
    a, _ = sac_model.predict(obs, deterministic=True); return a

def a2c_action(obs, env):
    a, _ = a2c_model.predict(obs, deterministic=True); return a

def td3_action(obs, env):
    a, _ = td3_model.predict(obs, deterministic=True); return a

_action_fns = {'PPO': ppo_action, 'SAC': sac_action, 'A2C': a2c_action, 'TD3': td3_action}


# ── Run all policies on the same test environment ──────────────────────────────
env_obs_size = make_test_env().observation_space.shape[0]
print(f'\nTest environment: {T_test} days | obs size: {env_obs_size}')
print('Running policies...')

results = {}

# SIP baseline — no model required, always runs
results['SIP'] = run_episode(make_test_env(), baseline_action)
print('  SIP  ✓')

# RL agents — only included if trained this session and obs layout matches
for name, model in _available.items():
    if check_model(model, env_obs_size, name):
        results[name] = run_episode(make_test_env(), _action_fns[name])
        print(f'  {name}  ✓')


# ── Aggregate cost metrics ────────────────────────────────────────────────────
def aggregate(rows):
    """
    Summarise one evaluation episode into a flat metrics dictionary.

    Metrics
    -------
    total_cost      total SEK spent over the test period
    annual_cost     total_cost normalised to 365 days
    isl             immediate service level = 1 - shortage / demand
    stockout_rate   fraction of test days on which any part was stocked out
    cost_per_demand total_cost / total_demand  (Jaffar / Fukuhara standard)
    n_orders        number of regular (non-urgent) orders placed
    n_rush_events   number of emergency replenishment events triggered
    end_backorders  units of unmet demand remaining at episode end
    """
    T_ep           = len(rows)
    total_cost     = sum(r['total_day_cost'] for r in rows)
    annual_cost    = total_cost / max(T_ep, 1) * 365.0
    total_demand   = sum(r[pt]['demand']   for r in rows for pt in part_types)
    total_shortage = sum(r[pt]['shortage'] for r in rows for pt in part_types)
    total_orders   = sum(1 for r in rows for pt in part_types if r[pt]['order_placed'])
    total_rush_ev  = sum(1 for r in rows for pt in part_types if r[pt]['urgent_qty'] > 0)
    total_order_qty= sum(r[pt]['order_qty'] for r in rows for pt in part_types)
    end_backorders = sum(rows[-1][pt]['backorders'] for pt in part_types) if rows else 0.0
    stockout_days  = sum(1 for r in rows if any(r[pt]['shortage'] > 0 for pt in part_types))
    breakdown = {
        'ordering': sum(r[pt]['ordering'] for r in rows for pt in part_types),
        'rush':     sum(r[pt]['rush']     for r in rows for pt in part_types),
        'holding':  sum(r[pt]['holding']  for r in rows for pt in part_types),
    }
    return {
        'total_cost':     total_cost,
        'annual_cost':    annual_cost,
        'isl':            1.0 - total_shortage / max(total_demand, 1.0),
        'total_demand':   total_demand,
        'total_shortage': total_shortage,
        'end_backorders': end_backorders,
        'urgent_units':   sum(r[pt]['urgent_qty'] for r in rows for pt in part_types),
        'n_orders':       total_orders,
        'n_rush_events':  total_rush_ev,
        'total_order_qty':total_order_qty,
        'breakdown':      breakdown,
        'stockout_days':  stockout_days,
        'stockout_rate':  stockout_days / max(T_ep, 1),
        'cost_per_demand': total_cost / max(total_demand, 1.0),
    }


metrics  = {name: aggregate(rows) for name, rows in results.items()}
pnames   = list(metrics.keys())
sip_cost = metrics['SIP']['total_cost']


# ── Results table ──────────────────────────────────────────────────────────────
print('\n' + '='*100)
print(f"  RESULTS — {T_test}-day held-out test period")
print('='*100)
print(f"{'Policy':<8} {'Total SEK':>10} {'Ann. SEK':>9} {'ISL':>6} {'vs SIP':>7}"
      f" {'RegOrd':>7} {'RushEv':>8} {'AvgQty':>8} {'EndBO':>7} {'StockoutRate':>13} {'CostPerDmd':>11}")
print('-'*100)
for n, m in metrics.items():
    ratio   = m['total_cost'] / sip_cost
    avg_qty = m['total_order_qty'] / max(m['n_orders'], 1)
    print(f"{n:<8} {m['total_cost']:>10,.0f} {m['annual_cost']:>9,.0f} {m['isl']:>6.3f}"
          f" {ratio:>7.2f}x {m['n_orders']:>7d} {m['n_rush_events']:>8d}"
          f" {avg_qty:>8.1f} {m['end_backorders']:>7.1f} {m['stockout_rate']:>13.4f} {m['cost_per_demand']:>11.4f}")
print('-'*100)
print('  RegOrd = regular orders placed | RushEv = automatic stockout-driven emergency orders')
print('  AvgQty = average units per regular order | EndBO = backorders outstanding at episode end')
print('  StockoutRate = fraction of days with any stockout | CostPerDmd = total cost / total demand units')

print(f"\nCost breakdown (SEK):")
print(f"{'Policy':<8} {'Ordering':>12} {'Rush':>12} {'Holding':>12} {'Total':>12}")
print('-'*52)
for n, m in metrics.items():
    b = m['breakdown']
    print(f"{n:<8} {b['ordering']:>12,.0f} {b['rush']:>12,.0f} {b['holding']:>12,.1f}"
          f" {m['total_cost']:>12,.0f}")
print('='*100)


# ── Save results to CSV ────────────────────────────────────────────────────────
import pandas as pd
rows_csv = []
for n, m in metrics.items():
    b = m['breakdown']
    rows_csv.append({
        'Policy':          n,
        'Total_Cost_SEK':  round(m['total_cost'], 2),
        'Annual_Cost_SEK': round(m['annual_cost'], 2),
        'ISL':             round(m['isl'], 4),
        'vs_SIP_ratio':    round(m['total_cost'] / sip_cost, 4),
        'N_Regular_Orders':m['n_orders'],
        'N_Rush_Events':   m['n_rush_events'],
        'Avg_Order_Qty':   round(m['total_order_qty'] / max(m['n_orders'], 1), 1),
        'Ordering_Cost':   round(b['ordering'], 2),
        'Rush_Cost':       round(b['rush'], 2),
        'Holding_Cost':    round(b['holding'], 4),
        'End_Backorders':  round(m['end_backorders'], 2),
        'Stockout_Rate':   round(m['stockout_rate'], 4),
        'Cost_Per_Demand': round(m['cost_per_demand'], 4),
    })
df_res = pd.DataFrame(rows_csv)
os.makedirs('../results', exist_ok=True)
df_res.to_csv('../results/policy_comparison.csv', index=False)
print('\nResults saved: results/policy_comparison.csv')


# ─────────────────────────────────────────────────────────────────────────────
# Plotting helpers
# ─────────────────────────────────────────────────────────────────────────────
try:
    plt.style.use('seaborn-v0_8-whitegrid')
except Exception:
    try:    plt.style.use('seaborn-whitegrid')
    except: pass

# Consistent colour palette — one colour per algorithm
COLORS = {
    'SIP': '#d62728',   # red
    'PPO': '#1f77b4',   # blue
    'SAC': '#2ca02c',   # green
    'A2C': '#ff7f0e',   # orange
    'TD3': '#9467bd',   # purple
}
x     = np.arange(len(pnames))
BAR_W = 0.55
bar_colors = [COLORS.get(n, '#888') for n in pnames]

SUFFIX  = '_2000_mask'
SAVEDIR = '../figures/Figures_2000_mask'
os.makedirs(SAVEDIR, exist_ok=True)

def save_fig(fig, stem):
    """Save as PDF + PNG with episode suffix, then close."""
    full = f'{stem}{SUFFIX}'
    fig.savefig(f'{SAVEDIR}/{full}.png', dpi=300, bbox_inches='tight')
    print(f'  Saved: {SAVEDIR}/{full}.png')
    plt.close(fig)
# Helper: add value labels above bars
def bar_labels(ax, bars, fmt='{:.0f}', offset_frac=0.01):
    ymax = max(b.get_height() for b in bars) if bars else 1
    for bar in bars:
        v = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                v + ymax * offset_frac,
                fmt.format(v),
                ha='center', va='bottom', fontsize=8)


print('\nSaving individual figures to results/figures/ ...')


# ── Figure 1: Learning curves ─────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
reward_map = {
    'PPO': globals().get('ep_rewards_ppo'),
    'SAC': globals().get('ep_rewards_sac'),
    'A2C': globals().get('ep_rewards_a2c'),
    'TD3': globals().get('ep_rewards_td3'),
}
for algo, rewards in reward_map.items():
    if rewards is None or len(rewards) == 0:
        continue
    ep_cost = [-r for r in rewards]
    w       = max(1, len(ep_cost) // 30)   # rolling-average window ≈ 3 % of training
    smooth  = np.convolve(ep_cost, np.ones(w) / w, mode='valid')
    ax.plot(ep_cost, color=COLORS[algo], alpha=0.12, lw=0.6)
    ax.plot(np.arange(len(smooth)) + (w - 1) / 2, smooth,
            color=COLORS[algo], lw=2.0, label=algo)
ax.set_xlabel('Episode', fontsize=11)
ax.set_ylabel('Episode cost (SEK)', fontsize=11)
ax.set_title('Learning Curves — Training Episode Cost', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
fig.tight_layout()
save_fig(fig, 'fig_learning_curves')


# ── Figure 2: Total test-period cost ─────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
vals = [metrics[n]['total_cost'] for n in pnames]
bars = ax.bar(x, vals, BAR_W, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:,.0f}')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Total cost (SEK)', fontsize=11)
ax.set_title(f'Total Cost — {T_test}-day Test Period', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
# Annotate vs-SIP ratios
sip_c = metrics['SIP']['total_cost']
for xi, n in zip(x, pnames):
    ratio = metrics[n]['total_cost'] / sip_c
    label = 'baseline' if n == 'SIP' else f'{ratio:.2f}× SIP'
    ax.text(xi, -max(vals)*0.06, label, ha='center', va='top', fontsize=7.5, color='#333')
ax.set_ylim(bottom=-max(vals)*0.12)
fig.tight_layout()
save_fig(fig, 'fig_total_cost')


# ── Figure 3: Cost breakdown (stacked bar) ───────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
ord_v = [metrics[n]['breakdown']['ordering'] for n in pnames]
rsh_v = [metrics[n]['breakdown']['rush']     for n in pnames]
hld_v = [metrics[n]['breakdown']['holding']  for n in pnames]
b1 = ax.bar(x, ord_v, BAR_W, label='Ordering cost',   color='#1f77b4', alpha=0.88)
b2 = ax.bar(x, rsh_v, BAR_W, bottom=ord_v,
            label='Rush (emergency) cost', color='#d62728', alpha=0.88)
b3 = ax.bar(x, hld_v, BAR_W, bottom=[o + r for o, r in zip(ord_v, rsh_v)],
            label='Holding cost',          color='#2ca02c', alpha=0.88)
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Cost (SEK)', fontsize=11)
ax.set_title('Cost Breakdown by Component', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))
fig.tight_layout()
save_fig(fig, 'fig_cost_breakdown')


# ── Figure 4: Immediate service level ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
vals_isl = [metrics[n]['isl'] for n in pnames]
bars = ax.bar(x, vals_isl, BAR_W, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, vals_isl):
    ax.text(bar.get_x() + bar.get_width() / 2,
            min(v + 0.002, 1.055), f'{v:.4f}',
            ha='center', va='bottom', fontsize=8)
ax.axhline(0.95, color='gray', ls='--', lw=1.2, label='Target ISL = 0.95')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylim(0.88, 1.06)
ax.set_ylabel('Immediate service level (ISL)', fontsize=11)
ax.set_title('Immediate Service Level (ISL)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig_isl')


# ── Figure 5: Cost per demand unit ───────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
cpd_vals = [metrics[n]['cost_per_demand'] for n in pnames]
bars = ax.bar(x, cpd_vals, BAR_W, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:.3f}')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Cost per demand unit (SEK)', fontsize=11)
ax.set_title('Cost Per Demand Unit', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.2f}'))
fig.tight_layout()
save_fig(fig, 'fig_cost_per_demand')


# ── Figure 6: Stockout rate ───────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
sr_vals = [metrics[n]['stockout_rate'] for n in pnames]
bars = ax.bar(x, sr_vals, BAR_W, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:.4f}')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Fraction of days with stockout', fontsize=11)
ax.set_title('Stockout Rate', fontsize=12, fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:.3f}'))
fig.tight_layout()
save_fig(fig, 'fig_stockout_rate')


# ── Figure 7: Regular order count vs rush event count ─────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
w2 = BAR_W / 2.2
ord_c  = [metrics[n]['n_orders']      for n in pnames]
rush_c = [metrics[n]['n_rush_events'] for n in pnames]
ax.bar(x - w2/2, ord_c,  w2, label='Regular orders', color='#1f77b4', alpha=0.88)
ax.bar(x + w2/2, rush_c, w2, label='Rush events',    color='#d62728', alpha=0.88)
for xi, ov, rv in zip(x, ord_c, rush_c):
    ax.text(xi - w2/2, ov + 0.5, str(ov), ha='center', va='bottom', fontsize=8)
    ax.text(xi + w2/2, rv + 0.5, str(rv), ha='center', va='bottom', fontsize=8)
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Number of events', fontsize=11)
ax.set_title('Regular Orders vs Emergency (Rush) Events', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig_order_vs_rush')


# ── Figure 8: Stock trajectory for the highest-demand part ────────────────────
_pt = part_types[int(np.argmax([test_demand_dict[pt].mean() for pt in part_types]))]
fig, ax = plt.subplots(figsize=(10, 4))
for n in pnames:
    ax.plot([r[_pt]['stock'] for r in results[n]],
            color=COLORS.get(n, '#888'), lw=1.3, alpha=0.85, label=n)
ax.axhline(0, color='black', lw=0.8, ls='--', label='Zero stock')
ax.set_xlabel('Day (test period)', fontsize=11)
ax.set_ylabel('On-hand stock (units)', fontsize=11)
ax.set_title(f'Stock Trajectory — part {_pt} (highest demand)', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig_stock_trajectory')


# ── Figure 9: Regular order timing for the same part ─────────────────────────
fig, ax = plt.subplots(figsize=(10, 4))
for n in pnames:
    order_days = [r['day'] for r in results[n] if r[_pt]['order_placed']]
    ax.scatter(order_days, [n] * len(order_days),
               color=COLORS.get(n, '#888'), marker='|', s=90, lw=1.8,
               label=f'{n} ({len(order_days)} orders)')
ax.set_xlabel('Day (test period)', fontsize=11)
ax.set_title(f'Regular Order Timing — part {_pt}', fontsize=12, fontweight='bold')
ax.set_yticks(range(len(pnames))); ax.set_yticklabels(pnames, fontsize=10)
ax.legend(fontsize=8, loc='upper right')
fig.tight_layout()
save_fig(fig, 'fig_order_timing')


# ── Figure 10: Average order quantity per policy ──────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
avg_qty_vals = [metrics[n]['total_order_qty'] / max(metrics[n]['n_orders'], 1) for n in pnames]
bars = ax.bar(x, avg_qty_vals, BAR_W, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:.1f}')
ax.axhline(5000, color='gray', ls='--', lw=1.2, label='Max order cap (5000 units)')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=10)
ax.set_ylabel('Average order quantity (units)', fontsize=11)
ax.set_title('Average Regular Order Quantity', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
fig.tight_layout()
save_fig(fig, 'fig_avg_order_qty')


# ── Combined overview (Figure 11): 2×3 grid for thesis appendix ──────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 11))
fig.subplots_adjust(hspace=0.45, wspace=0.38)

# (a) Learning curves
ax = axes[0, 0]
for algo, rewards in reward_map.items():
    if rewards is None or len(rewards) == 0:
        continue
    ep_cost = [-r for r in rewards]
    w       = max(1, len(ep_cost) // 30)
    smooth  = np.convolve(ep_cost, np.ones(w) / w, mode='valid')
    ax.plot(ep_cost, color=COLORS[algo], alpha=0.12, lw=0.6)
    ax.plot(np.arange(len(smooth)) + (w - 1) / 2, smooth, color=COLORS[algo], lw=2, label=algo)
ax.set_xlabel('Episode'); ax.set_ylabel('Episode cost (SEK)')
ax.set_title('(a) Learning Curves', fontweight='bold')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# (b) Total cost
ax = axes[0, 1]
vals = [metrics[n]['total_cost'] for n in pnames]
bars = ax.bar(x, vals, BAR_W, color=bar_colors, alpha=0.85)
bar_labels(ax, bars, fmt='{:,.0f}')
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=9)
ax.set_ylabel('Total cost (SEK)')
ax.set_title('(b) Test Total Cost', fontweight='bold')
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# (c) ISL
ax = axes[0, 2]
vals_isl = [metrics[n]['isl'] for n in pnames]
bars = ax.bar(x, vals_isl, BAR_W, color=bar_colors, alpha=0.85)
for bar, v in zip(bars, vals_isl):
    ax.text(bar.get_x() + bar.get_width() / 2, min(v + 0.002, 1.08),
            f'{v:.3f}', ha='center', va='bottom', fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=9)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Service level (ISL)')
ax.set_title('(c) Immediate Service Level', fontweight='bold')
ax.axhline(0.95, color='gray', ls='--', lw=1, label='Target 0.95')
ax.legend(fontsize=8)

# (d) Cost breakdown
ax = axes[1, 0]
ax.bar(x, ord_v, BAR_W, label='Ordering', color='#1f77b4', alpha=0.85)
ax.bar(x, rsh_v, BAR_W, bottom=ord_v, label='Rush', color='#d62728', alpha=0.85)
ax.bar(x, hld_v, BAR_W, bottom=[o + r for o, r in zip(ord_v, rsh_v)],
       label='Holding', color='#2ca02c', alpha=0.85)
ax.set_xticks(x); ax.set_xticklabels(pnames, fontsize=9)
ax.set_ylabel('Cost (SEK)')
ax.set_title('(d) Cost Breakdown', fontweight='bold')
ax.legend(fontsize=8)
ax.yaxis.set_major_formatter(plt.FuncFormatter(lambda v, _: f'{v:,.0f}'))

# (e) Stock trajectory
ax = axes[1, 1]
for n in pnames:
    ax.plot([r[_pt]['stock'] for r in results[n]],
            color=COLORS.get(n, '#888'), lw=1.2, alpha=0.85, label=n)
ax.axhline(0, color='black', lw=0.8, ls='--')
ax.set_xlabel('Day (test period)')
ax.set_ylabel('On-hand stock')
ax.set_title(f'(e) Stock Trajectory — {_pt}', fontweight='bold')
ax.legend(fontsize=8)

# (f) Order timing
ax = axes[1, 2]
for n in pnames:
    order_days = [r['day'] for r in results[n] if r[_pt]['order_placed']]
    ax.scatter(order_days, [n] * len(order_days),
               color=COLORS.get(n, '#888'), marker='|', s=80, lw=1.5)
ax.set_xlabel('Day (test period)')
ax.set_title(f'(f) Regular Order Timing — {_pt}', fontweight='bold')
ax.set_yticks(range(len(pnames))); ax.set_yticklabels(pnames, fontsize=9)

sip_ratio_str = '  |  '.join(
    f"{n}: {metrics[n]['total_cost'] / sip_cost:.2f}× SIP"
    for n in pnames if n != 'SIP'
)
plt.suptitle(
    f"RL vs SIP (s,Q) — {T_test}-day held-out test period\n{sip_ratio_str}",
    fontsize=11, fontweight='bold',
)
plt.tight_layout()
save_fig(fig, 'fig_overview_6panel')


# ── Combined overview (Figure 12): 1×3 inventory metrics ─────────────────────
fig2, axes2 = plt.subplots(1, 3, figsize=(16, 5))
fig2.subplots_adjust(wspace=0.4)

ax = axes2[0]
cpd_vals = [metrics[n]['cost_per_demand'] for n in pnames]
bars = ax.bar(pnames, cpd_vals, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:.3f}')
ax.set_ylabel('Cost per demand unit (SEK)', fontsize=10)
ax.set_title('Cost Per Demand', fontweight='bold')

ax = axes2[1]
sr_vals = [metrics[n]['stockout_rate'] for n in pnames]
bars = ax.bar(pnames, sr_vals, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
bar_labels(ax, bars, fmt='{:.4f}')
ax.set_ylabel('Fraction of days with stockout', fontsize=10)
ax.set_title('Stockout Rate', fontweight='bold')

ax = axes2[2]
isl_vals = [metrics[n]['isl'] for n in pnames]
bars = ax.bar(pnames, isl_vals, color=bar_colors, alpha=0.88, edgecolor='white', linewidth=0.5)
for bar, v in zip(bars, isl_vals):
    ax.text(bar.get_x() + bar.get_width() / 2,
            min(v + 0.002, 1.06), f'{v:.4f}',
            ha='center', va='bottom', fontsize=8)
ax.set_ylim(0.90, 1.05)
ax.set_ylabel('Immediate service level (ISL)', fontsize=10)
ax.set_title('Service Level (ISL)', fontweight='bold')
ax.axhline(0.95, color='gray', ls='--', lw=1, label='Target 0.95')
ax.legend(fontsize=8)

fig2.suptitle(
    f'Inventory Performance Metrics — {T_test}-day held-out test period',
    fontsize=12, fontweight='bold',
)
plt.tight_layout()
save_fig(fig2, 'fig_inventory_metrics_3panel')


print('\nAll figures saved to results/figures/')
print('Individual PDFs for LaTeX:')
for stem in ['fig_learning_curves', 'fig_total_cost', 'fig_cost_breakdown',
             'fig_isl', 'fig_cost_per_demand', 'fig_stockout_rate',
             'fig_order_vs_rush', 'fig_stock_trajectory', 'fig_order_timing',
             'fig_avg_order_qty', 'fig_overview_6panel', 'fig_inventory_metrics_3panel']:
    print(f'  \\includegraphics[width=\\linewidth]{{figures/{stem}}}')


Models available for evaluation:
  PPO: trained this session ✓
  SAC: trained this session ✓
  A2C: trained this session ✓
  TD3: trained this session ✓

Test environment: 439 days | obs size: 52
Running policies...
  SIP  ✓
  PPO  ✓
  SAC  ✓
  A2C  ✓
  TD3  ✓

  RESULTS — 439-day held-out test period
Policy    Total SEK  Ann. SEK    ISL  vs SIP  RegOrd   RushEv   AvgQty   EndBO  StockoutRate  CostPerDmd
----------------------------------------------------------------------------------------------------
SIP           1,255     1,044  0.998    1.00x       4        2   2950.0     0.0        0.0046      0.2731
PPO          22,157    18,422  0.939   17.65x       0      103      0.0     0.0        0.2187      4.8209
SAC           1,373     1,142  0.998    1.09x       5        2   2429.2     0.0        0.0046      0.2988
A2C          33,781    28,087  0.931   26.91x      56      118     25.5     8.0        0.2437      7.3501
TD3           1,255     1,044  0.998    1.00x       4        2   29